# FDMNES XAS simulation workflow with Larixite
This notebook shows an example workflow for FDMNES XAS simulation using Larixite.

- Author: Mauro Rovezzi
- Contact: mauro.rovezzi@esrf.fr
- Status: *in progress...*
- Last update: 2026-05-12

First import the necessary modules and set some variables

In [2]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import crystal_toolkit
import warnings
from larixite.struct import get_structs_from_dir
from larixite.fdmnes import FdmnesXasInput
from larixite.fdmnes import logger

logger.setLevel("INFO")  #: adjust this level if you want to have more or less information: "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"
warnings.simplefilter('always', UserWarning)  #: to filter out some technic warnings from underlying pymatgen

#: set a directory containing the structural files (TODO: set your own path)
curdir = Path().cwd()
basedir = curdir.parent
testdir = basedir / "tests"
structsdir = testdir / "structs"

abs = "Zn"  #: select the absorber

fdmnes_executable = "fdmnes"  #: path to fdmnes executable (TODO: set your own path)

Get a list of XasStructure objects from a directory containing structural files, based on the absorbing element specified before.

In [3]:
structs = get_structs_from_dir(structsdir, abs, globstr=f"*{abs}*")

/home/esrf/rovezzi/devel/larixite/larixite/struct/__init__.py:88: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = structs.parse_structures(primitive=False)[frame]
[larixite.struct | INFO    ] 0: ZnO_wurtzite_Weber1923_COD-1011259.cif
/home/esrf/rovezzi/devel/larixite/larixite/struct/__init__.py:88: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = structs.parse_structures(primitive=False)[frame]
[larixite.struct | INFO    ] 1: ZnO_mp-2133.cif
/home/esrf/rovezzi/devel/larixite/larixite/struct/__init__.py:88: UserWarning: Issues encountered while parsing CIF: 32 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = structs.parse_structures(primitive=False)[frame]
[larixite.struct | INFO    ] 2: ZnO_mp-997630.cif


select one structure and print some information about it

In [4]:
sg = structs[0]
sg.show_unique_sites()

Unique sites for ZnO_wurtzite_Weber1923_COD-1011259.cif
iunique	istruct	label	frac_coords                       	occu	cart_coords                                   	nsites	wyckoff	species_string
0 (abs)	0      	Zn1  	[0.33333333 0.66666667 0.        ]	1   	[3.58065429e-16 1.93470075e+00 2.05189571e-16]	2     	2b     	Zn2+          
1      	2      	O1   	[0.33333333 0.66666667 0.375     ]	1   	[3.58065429e-16 1.93470075e+00 1.95975000e+00]	2     	2b     	O2-           


the structure can be visualized with the [Crystal Toolkit](https://docs.crystaltoolkit.org/)

In [5]:
sg.struct

The absorber site is set by default to the first site with the absorbing element, but can be changed by indicating the index in the structure (`istruct` given by the `show_unique_sites()` method) to the `absorber_idx` attribute. It is convenient to check with the `show_unique_sites()` method that the index of the absorbing site is correctly set, as it gets the `(abs)` label next to the index.

In [6]:
sg.absorber_idx = 0
sg.show_unique_sites()

Unique sites for ZnO_wurtzite_Weber1923_COD-1011259.cif
iunique	istruct	label	frac_coords                       	occu	cart_coords                                   	nsites	wyckoff	species_string
0 (abs)	0      	Zn1  	[0.33333333 0.66666667 0.        ]	1   	[3.58065429e-16 1.93470075e+00 2.05189571e-16]	2     	2b     	Zn2+          
1      	2      	O1   	[0.33333333 0.66666667 0.375     ]	1   	[3.58065429e-16 1.93470075e+00 1.95975000e+00]	2     	2b     	O2-           


then generate an FDMNES input file FdmnesXasInput object and show it

In [7]:
f = FdmnesXasInput(sg, absorber=abs, struct_type="crystal", radius=5, erange="-5 0.5 10", optimize=True)
text = f.get_input()
print(text)

! FDMNES input file generated by larixite
! 2026-05-12 16:00:04
! larixite : 2025.5.1
! pymatgen : 2025.6.14
Header
Python
Comment
   ZnO_wurtzite_Weber1923_COD-1011259.cif: Zn (30) K edge
Filout
   job
Edge
   K
!<Energy range>
Range
   -5 0.5 10
! Energpho !output energy as photon energy (default relative to Fermi level)
Radius
   5.00
!<Multiple scattering mode>
Green
!<Multipolar expansion>
Quadrupole
!<Polarization and dichroism>
! Polarize 
!  =>TODO
! TDDFT
Memory_save
! Relativism
! Spinorbit
! Density
! Density_all
!<Exchange-correlation potential>
! Hedin !PBE96 is default since March 2026
!<Optimisation of self-consistency>
! SpGr_atom !`Full_atom` is default since March 2026
!<Self-consistent calculations>
! SCF
R_self
   3.5
N_self
   100
P_self
   0.025
! SCF_exc
! Vmax
! Atom_conf
!<Structure description with absorber: crystal>
Z_absorber
   30
Spgroup
   186
Occupancy
Crystal
   3.351 3.351 5.226 90.0 90.0 119.99999999999999
 30    0.3333333333    0.6666666667    0.0000

write the FDMNES input file and run FDMNES

In [13]:
outdir = f.write_input(outdir="/tmp_14_days/larixite")

[larixite.fdmnes | INFO    ] written `/tmp_14_days/larixite/job.inp`


In [19]:
f.write_sbatch()

[larixite.fdmnes | INFO    ] written /tmp_14_days/larixite/job.sbatch


In [9]:
! module purge && module load fdmnes && cd {outdir.as_posix()} && {fdmnes_executable}


 FDMNES program, Revision 3rd of October 2025      
   Date = 12 05 2026                              
   Time = 14 h 00 mn 05 s                         

   python   
   edge     
   range    
   radius   
   green    
   quadrupol
   memory_sa
   r_self   
   n_self   
   p_self   
   z_absorbe
   spgroup  
   occupancy
   crystal  

 Filout: job
 Threshold: Zinc K1 edge

 Sequential calculation

 Number of calculated non equivalent absorbing atom =    1

 E_edge     =  9659.00 eV

 Cluster radius = 5.00 A, nb. of atom =  42
 Fermi energy calculation : cluster radius = 3.50 A, nb. of atom =  18
 Potential sup calculation: cluster radius = 7.50 A, nb. of atom = 144

 Point group : 3m       (C3v  )

 Point group used : 3        (C3   )

  ia   Z  mult     ch_val    ch_core   ch_total     ch_out   Atom charge
   1  30    1      11.520     16.998     28.518      0.001      1.482
   2   8    1       5.374      2.000      7.374      0.004      0.626
   3   8    3       5.372      2.000    

The following steps will be showed in the next version of Larixite:

- load the FDMNES output files
- plot the XAS spectrum
- compare the simulated spectrum with experimental data and adjust the convolution parameters
- run multiple jobs for convergence studies